# Part B: reasoning-chain SFT (distillation) on Kaggle

A small, self-contained reasoning fine-tune to make Part B concrete: teach a model
to produce an explicit `<think> ... </think>` reasoning trace before its answer, by
SFT on examples that already contain traces.

What this demonstrates (the brief's Part B):
- **Trace format**: the assistant target is `<think>reasoning</think>` then the answer.
- **Distillation**: we SFT on (question, reasoning, answer) triples. Here the traces
  come ready-made from GSM8K (real step-by-step solutions); in a real pipeline a
  strong teacher model would generate them and you'd keep only the correct ones.
- **The 75/25 mix**: ~75% reasoning data (GSM8K) + ~25% ordinary instructions
  (Alpaca), so the model keeps general ability and does not only think.

No dataset upload needed, it pulls GSM8K + Alpaca from Hugging Face. Just turn on
**Settings -> Accelerator -> GPU T4** and Run All. ~15-25 min on a T4.


In [ ]:
!pip -q install -U "transformers>=4.44" "trl>=0.9" peft bitsandbytes datasets accelerate
!pip -q uninstall -y torchao   # Kaggle ships an old torchao that recent peft rejects; we use bitsandbytes
print("deps installed")

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # swap to "Qwen/Qwen3-1.7B" for native thinking mode (bigger)
OUT = "/kaggle/working"
N_REASON, N_NONREASON = 750, 250            # 75/25 reasoning : non-reasoning
MAX_LEN = 1024                              # reasoning traces are longer than a label
print("config set")

## Build the reasoning dataset (GSM8K traces + Alpaca, 75/25)

In [ ]:
from datasets import load_dataset, concatenate_datasets

# Reasoning data: GSM8K answers are "<step by step> #### <final>". We wrap the steps
# in <think>...</think> so the assistant target is trace-then-answer.
def gsm8k_to_messages(ex):
    reasoning, _, final = ex["answer"].partition("####")
    content = f"<think>\n{reasoning.strip()}\n</think>\n\nThe answer is {final.strip()}."
    return {"messages": [{"role": "user", "content": ex["question"]},
                         {"role": "assistant", "content": content}]}

# Non-reasoning 25%: plain instruction/response, no <think> block.
def alpaca_to_messages(ex):
    user = ex["instruction"] + (("\n\n" + ex["input"]) if ex.get("input") else "")
    return {"messages": [{"role": "user", "content": user},
                         {"role": "assistant", "content": ex["output"]}]}

reason = load_dataset("gsm8k", "main", split=f"train[:{N_REASON}]")
reason = reason.map(gsm8k_to_messages, remove_columns=reason.column_names)
nonr = load_dataset("tatsu-lab/alpaca", split=f"train[:{N_NONREASON}]")
nonr = nonr.map(alpaca_to_messages, remove_columns=nonr.column_names)

print("=== a reasoning example (note the <think> trace) ===")
print(reason[0]["messages"][1]["content"][:400])
print("\n=== a non-reasoning example (no trace) ===")
print(nonr[0]["messages"][1]["content"][:200])

mix = concatenate_datasets([reason, nonr]).shuffle(seed=0)
print(f"\nmix: {len(mix)} rows = {N_REASON} reasoning + {N_NONREASON} non-reasoning (75/25)")

## Distillation SFT: train the model to emit `<think>` traces

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=quant,
    torch_dtype=torch.bfloat16, device_map="auto")
model.config.use_cache = False
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules="all-linear")
out = f"{OUT}/lora-reasoning"
cfg = SFTConfig(output_dir=out, num_train_epochs=2, per_device_train_batch_size=2,
    gradient_accumulation_steps=8, learning_rate=2e-4, lr_scheduler_type="cosine",
    warmup_ratio=0.05, logging_steps=10, save_strategy="epoch", bf16=True,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=MAX_LEN, packing=False, assistant_only_loss=True, seed=0, report_to="none")
trainer = SFTTrainer(model=model, args=cfg, train_dataset=mix, peft_config=lora, processing_class=tok)
r = trainer.train(); trainer.save_model(out); tok.save_pretrained(out)
print(f"final_train_loss={r.training_loss:.4f} -> {out}")

## Before vs after: does it reason now?

In [ ]:
# Before/after: does the fine-tune now produce a <think> trace then an answer?
from datasets import load_dataset
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_model(adapter=None):
    t = AutoTokenizer.from_pretrained(BASE_MODEL); t.padding_side = "left"
    if t.pad_token is None: t.pad_token = t.eos_token
    m = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16)
    if adapter:
        from peft import PeftModel; m = PeftModel.from_pretrained(m, adapter)
    return m.to("cuda").eval(), t

@torch.no_grad()
def ask(m, t, q, max_new=256):
    txt = t.apply_chat_template([{"role":"user","content":q}], add_generation_prompt=True, tokenize=False)
    enc = t(txt, return_tensors="pt", add_special_tokens=False).to(m.device)
    g = m.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=t.pad_token_id)
    return t.decode(g[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

QS = load_dataset("gsm8k", "main", split="test[:3]")["question"]
mb, tb = load_model(None)
for q in QS: print("Q:", q, "\nBASE  :", ask(mb, tb, q)[:500], "\n")
del mb; torch.cuda.empty_cache()
mt, tt = load_model(f"{OUT}/lora-reasoning")
for q in QS: print("Q:", q, "\nTUNED :", ask(mt, tt, q)[:500], "\n")

## Download the adapter

In [ ]:
import shutil
shutil.make_archive(f"{OUT}/lora-reasoning", "zip", f"{OUT}/lora-reasoning")
print("Download lora-reasoning.zip from the Output tab.")